# Lesson 4: Building a Multi-Document Agent

Extend the research agent across multiple papers with Gemini + LlamaIndex, then scale with **tool retrieval** (`ObjectIndex`).

## Setup

In [ ]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## 1. Setup an agent over 3 papers

Place these PDFs in `week_3/` (or download with the URLs below):

- `metagpt.pdf`
- `longlora.pdf`
- `selfrag.pdf`

In [ ]:
urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",
    "https://openreview.net/pdf?id=6PmJoRfdaK",
    "https://openreview.net/pdf?id=hSyW5go0v8",
]

papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "selfrag.pdf",
]

# Optional download:
# for url, paper in zip(urls, papers):
#     !curl -L "{url}" -o "{paper}"

In [ ]:
from pathlib import Path

from utils import get_doc_tools

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

In [ ]:
initial_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [ ]:
from llama_index.llms.google_genai import GoogleGenAI
from helper import DEFAULT_LLM_MODEL

llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)

In [ ]:
len(initial_tools)

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context

agent = FunctionAgent(
    tools=initial_tools,
    llm=llm,
    system_prompt=(
        "You are an agent designed to answer queries over a set of given papers. "
        "Please always use the tools provided to answer a question. "
        "Do not rely on prior knowledge."
    ),
    verbose=True,
)
ctx = Context(agent)

In [ ]:
response = await agent.run(
    "Tell me about the evaluation dataset used in LongLoRA, "
    "and then tell me about the evaluation results",
    ctx=ctx,
)
print(str(response))

In [ ]:
response = await agent.run(
    "Give me a summary of both Self-RAG and LongLoRA",
    ctx=ctx,
)
print(str(response))

## 2. Setup an agent over 11 papers

### Download 11 ICLR papers

When scaling to many tools, pass a **tool retriever** instead of all tools at once.

In [ ]:
urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",
    "https://openreview.net/pdf?id=6PmJoRfdaK",
    "https://openreview.net/pdf?id=LzPWWPAdY4",
    "https://openreview.net/pdf?id=VTF8yNQM66",
    "https://openreview.net/pdf?id=hSyW5go0v8",
    "https://openreview.net/pdf?id=9WD9KwssyT",
    "https://openreview.net/pdf?id=yV6fD7LYkF",
    "https://openreview.net/pdf?id=hnrB5YHoYu",
    "https://openreview.net/pdf?id=WbWtOYIzIK",
    "https://openreview.net/pdf?id=c5pwL0Soay",
    "https://openreview.net/pdf?id=TpD2aG1h0D",
]

papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "loftq.pdf",
    "swebench.pdf",
    "selfrag.pdf",
    "zipformer.pdf",
    "values.pdf",
    "finetune_fair_diffusion.pdf",
    "knowledge_card.pdf",
    "metra.pdf",
    "vr_mcl.pdf",
]

# Optional download:
# for url, paper in zip(urls, papers):
#     !curl -L "{url}" -o "{paper}"

In [ ]:
from pathlib import Path

from utils import get_doc_tools

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

### Extend the Agent with Tool Retrieval

In [ ]:
all_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(
    all_tools,
    index_cls=VectorStoreIndex,
)

In [ ]:
obj_retriever = obj_index.as_retriever(similarity_top_k=3)

In [ ]:
tools = obj_retriever.retrieve(
    "Tell me about the eval dataset used in MetaGPT and SWE-Bench"
)
for t in tools:
    print(t.metadata)

In [ ]:
agent = FunctionAgent(
    tool_retriever=obj_retriever,
    llm=llm,
    system_prompt="""\
You are an agent designed to answer queries over a set of given papers.
Please always use the tools provided to answer a question. Do not rely on prior knowledge.
""",
    verbose=True,
)
ctx = Context(agent)

In [ ]:
response = await agent.run(
    "Tell me about the evaluation dataset used "
    "in MetaGPT and compare it against SWE-Bench",
    ctx=ctx,
)
print(str(response))

In [ ]:
response = await agent.run(
    "Compare and contrast the LoRA papers (LongLoRA, LoftQ). "
    "Analyze the approach in each paper first.",
    ctx=ctx,
)
print(str(response))